# tocandira, round 2 of 2: train the evaluation net

After round 1 finished. Loads every shard in the data folder, holds out 5% of the games,
trains the NNUE style net once and reports, per epoch, the same metrics for nokamute's hand
tuned static evaluation (`baseline`) and for the net. The net is only interesting once it
beats the baseline on held out games.

L4 runtime (GPU, about 1.7 units/hour; a 256 wide net does an epoch over a few million
positions in minutes). To run two experiments at once, save a copy in Drive per run
(one notebook file per runtime), each with its own `RUN_NAME` and hyperparameters; each run
writes to its own folder.

In [ ]:
RUN_NAME = "h512"
DATA_DIR = "/content/drive/MyDrive/tocandira/data/d4"
OUT_DIR = f"/content/drive/MyDrive/tocandira/models/{RUN_NAME}"
HIDDEN, L2 = 512, 32
EPOCHS, BATCH, LR = 20, 4096, 1e-3
LAMBDA = 0.75           # weight of the search score in the target; the rest is the game outcome

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then curl -sSf https://sh.rustup.rs | sh -s -- -y -q >/dev/null; fi
source "$HOME/.cargo/env"
cd /content
# Cloned to /content/repo, not /content/tocandira: a folder named like the package would shadow it on sys.path.
if [ -d repo ]; then (cd repo && git pull -q && git submodule update --init -q); else git clone -q --recurse-submodules https://github.com/abstratovcm/tocandira repo; fi
cd /content/repo/engine/nokamute
CARGO_PROFILE_RELEASE_LTO=true CARGO_PROFILE_RELEASE_CODEGEN_UNITS=1 cargo build --release -q
# Not an editable install: its .pth file would only be read when the kernel restarts.
cd /content/repo && pip install -q .
./engine/nokamute/target/release/nokamute --help | head -1

In [ ]:
import glob

import numpy as np

from tocandira.nnue import Config, load_shards, train

paths = sorted(glob.glob(f"{DATA_DIR}/*.npz"))
data = load_shards(paths)
print(f"{len(paths)} shards, {len(data['score'])} positions, {len(np.unique(data['game']))} games")

In [ ]:
cfg = Config(hidden=HIDDEN, l2=L2, epochs=EPOCHS, batch=BATCH, lr=LR, lam=LAMBDA)
model = train(data, cfg, OUT_DIR)

In [ ]:
import json

import pandas as pd

rows = [json.loads(line) for line in open(f"{OUT_DIR}/log.jsonl")]
baseline = rows[0]["baseline"]
table = pd.DataFrame(rows[1:]).set_index("epoch")
print("baseline (static eval):", baseline)
table

In [ ]:
ax = table[["loss"]].plot(title="validation loss (lower is better)")
ax.axhline(baseline["loss"], color="gray", linestyle="--", label="static eval")
ax.legend()

What to look at: `loss` (fit to the blended target), `pearson` (agreement with the depth 4
score) and `sign_agreement` (does the sign predict the winner on decided positions). The
baseline row is what 344 lines of hand tuned Rust achieve. If the net does not clear it,
more shards or a wider net (`HIDDEN`) come first; if it does, `weights.npz` in `OUT_DIR` is
what the engine side will load.